In [ ]:
import sys
sys.path.insert(0, "/global/homes/l/lettieri/gcr-catalogs")
sys.path.insert(0, "../match_by_proximity")
sys.path.insert(0, "../")

from astropy.io import fits
from astropy.table import unique, Table
from GCR import GCRQuery
import GCRCatalogs
from generate_mock import generate_halo_cluster, HOD_model, generate_halo_galaxies, generate_cluster_galaxies, sharing_galaxies
from matplotlib import pyplot as plt
import numpy as np
from numcosmo_py import Ncm, Nc
from numcosmo_py.external.pyssc import pyssc as PySS
from numcosmo_py.helper import npa_to_seq
from numcosmo_py import sky_match
from numcosmo_py.plotting.tools import set_rc_params_article, confidence_ellipse
import pandas as pd
import time
from time_model import PySLineModel, PySLineGauss
from tqdm import tqdm
#set_rc_params_article(ncol=1)



plt.rcParams.update({
    "font.family": "serif",
    "mathtext.fontset": "cm",  # This tells Matplotlib to use Computer Modern for math
    "axes.labelweight": "normal"
})

print('GCRCatalogs =', GCRCatalogs.__version__, '|' ,'GCR =', GCRCatalogs.GCR.__version__)

Ncm.cfg_init()
%load_ext autoreload
%autoreload 2    

In [ ]:
Omega_b = 0.0486
Omega_c = 0.2614
Omega_k = 0.0
H0 = 67.7

#Omega_b = 0.05
#Omega_c = 0.25
#Omega_k = 0.0
#H0 = 70.0

# Create a cosmology object
cosmo = Nc.HICosmoDEXcdm.new()
cosmo.omega_x2omega_k()
cosmo["Omegab"] = Omega_b
cosmo["Omegac"] = Omega_c
cosmo["Omegak"] = Omega_k
cosmo["H0"] = H0
cosmo["w"] = -1.0

dist = Nc.Distance.new(100.0)
dist.compute_inv_comoving(True)
dist.prepare(cosmo)

# 2D match time test

In [ ]:
alphas_q = []
alphas_err_q = []
a_s_q = []
a_s_err_q = []

alphas_m = []
alphas_err_m = []
a_s_m = []
a_s_err_m = []


halo_coordinates = {"RA":"RA" , "DEC":"DEC" , "z":"z"}
halo_properties  = {"Mass":"halo_mass","halo_id":"halo_id", "R200":"R200"}
detections_coordinates =  {"RA":"RA" , "DEC":"DEC" , "z":"z"}
cluster_properties  = {"Mass":"cluster_mass","cluster_id":"cluster_id", "R200":"R200"}

sizes_halo = []
fig_q, ax_q = plt.subplots(figsize=(5, 3))
fig_m, ax_m = plt.subplots(figsize=(5, 3))

mean_time_halo_q = []
mean_time_halo_m = []
mean_time_halo = []
std_time_halo_q  = []
std_time_halo_m  = []
std_time_halo = []
sizes_halo = np.array([35000, 50000, 75000, 100000, 250000, 350000, 500000, 750000, 1000000])
knearests  = [10,20,40,60,80,100]

#sizes_halo = np.array([35000, 50000, 75000])
#knearests  = [10,20]

pbar_k = tqdm(knearests, desc="Processing k Nearest neighboors", unit="k")

for k in knearests:
    pbar_k.set_postfix({"Current k": f"{k}"})
    mean_time_halo_q_temp = []
    mean_time_halo_m_temp = []
    mean_time_halo_temp = []
    std_time_halo_q_temp  = []
    std_time_halo_m_temp  = []
    std_time_halo_temp = []
    size_halo_temp = []
    
    for size_halo in sizes_halo:
        reduced_halos, clusters = generate_halo_cluster(cosmo,dist,size_halo,30000)
        print("Mock data generated")
        # Update the progress bar suffix with the current mass value
    
        halos = sky_match.SkyMatch(
            query_data=reduced_halos, 
            query_coordinates=halo_coordinates,
            match_data=clusters,
            match_coordinates=detections_coordinates
        )
    
        detections = halos.invert_query_match()
        time_match_q, time_match_m, time_match = [], [], []
        for i in range(5):
            start = time.perf_counter()
            # Match phase
            start_q = time.perf_counter()
            halos_matched = halos.match_2d(cosmo, k, distance_method=sky_match.DistanceMethod.QUERY_RADIUS)
            
            mask_halos = (halos_matched.filter_mask_by_redshift_proximity(sigma_z=0.01) & 
                          halos_matched.filter_mask_by_distance(1))
            unique_halos = halos_matched.select_best(mask=mask_halos, selection_criteria=sky_match.SelectionCriteria.MORE_MASSIVE, more_massive_column='Mass')
            end_q = time.perf_counter()
            time_match_q.append(end_q - start_q) 
    
    
            # Detection match phase
            start_m = time.perf_counter()
            detections_matched = detections.match_2d(cosmo, k, distance_method=sky_match.DistanceMethod.MATCH_RADIUS)

            mask_detections = (detections_matched.filter_mask_by_redshift_proximity(sigma_z=0.01) & 
                               detections_matched.filter_mask_by_distance(1))
    
            unique_detections = detections_matched.select_best(mask=mask_detections, selection_criteria=sky_match.SelectionCriteria.MORE_MASSIVE, more_massive_column='Mass')
            end_m = time.perf_counter()
            time_match_m.append(end_m - start_m)
            cross_indices = unique_halos.get_cross_match_indices(unique_detections)
            
      
    
            end = time.perf_counter()
            time_match.append(end - start)
    
        time_match_q = np.array(time_match_q)
        time_match_m = np.array(time_match_m)
        time_match   = np.array(time_match)
        
        mean_time_halo_q_temp.append(np.mean(time_match_q))
        std_time_halo_q_temp.append(np.std(time_match_q))
    
    
        mean_time_halo_m_temp.append(np.mean(time_match_m))
        std_time_halo_m_temp.append(np.std(time_match_m))
    
        mean_time_halo_temp.append(np.mean(time_match))
        std_time_halo_temp.append(np.std(time_match))
    
    mean_time_halo_q_temp = np.array(mean_time_halo_q_temp)
    std_time_halo_q_temp = np.array(std_time_halo_q_temp)
    

    mean_time_halo_m_temp = np.array(mean_time_halo_m_temp)
    std_time_halo_m_temp = np.array(std_time_halo_m_temp)
    
    
    mean_time_halo_temp = np.array(mean_time_halo_temp)
    std_time_halo_temp = np.array(std_time_halo_temp)

    slm_q = PySLineModel()
    slm_q.props.alpha = 0.9
    slm_q.props.a = 0.2

    slm_m = PySLineModel()
    slm_m.props.alpha = 0.9
    slm_m.props.a = 0.2
    
    
    mset_q = Ncm.MSet.empty_new()
    mset_q.set(slm_q)
    mset_q.param_set_all_ftype(Ncm.ParamType.FREE)
    mset_q.prepare_fparam_map()

    mset_m = Ncm.MSet.empty_new()
    mset_m.set(slm_m)
    mset_m.param_set_all_ftype(Ncm.ParamType.FREE)
    mset_m.prepare_fparam_map()
    
    cov_q = Ncm.Matrix.new(len(mean_time_halo_q_temp), len(mean_time_halo_q_temp))
    cov_q.set_zero()
    cov_q.set_diag(Ncm.Vector.new_array(npa_to_seq(np.array(std_time_halo_q_temp)/100)))

    cov_m = Ncm.Matrix.new(len(mean_time_halo_m_temp), len(mean_time_halo_m_temp))
    cov_m.set_zero()
    cov_m.set_diag(Ncm.Vector.new_array(npa_to_seq(np.array(std_time_halo_m_temp)/100)))
    
    sld_q = PySLineGauss()
    sld_q.set_size(len(mean_time_halo_q_temp))
    sld_q.set_init(True)
    sld_q.use_norma(False)   
    sld_q.xv = Ncm.Vector.new_array(npa_to_seq(sizes_halo/1e6))
    sld_q.set_cov(cov_q)
    sld_q.peek_mean().set_array(npa_to_seq(mean_time_halo_q_temp/100))

    sld_m = PySLineGauss()
    sld_m.set_size(len(mean_time_halo_m_temp))
    sld_m.set_init(True)
    sld_m.use_norma(False)   
    sld_m.xv = Ncm.Vector.new_array(npa_to_seq(sizes_halo/1e6))
    sld_m.set_cov(cov_m)
    sld_m.peek_mean().set_array(npa_to_seq(mean_time_halo_m_temp/100))
    
    
    dset_q = Ncm.Dataset.new()
    dset_q.append_data(sld_q)
    lh_q = Ncm.Likelihood.new(dset_q)

    dset_m = Ncm.Dataset.new()
    dset_m.append_data(sld_m)
    lh_m = Ncm.Likelihood.new(dset_m)
    
    fit_q = Ncm.Fit.factory(Ncm.FitType.NLOPT, "ln-neldermead", lh_q, mset_q, Ncm.FitGradType.NUMDIFF_FORWARD)
    fit_q.run(Ncm.FitRunMsgs.FULL)
    fit_q.log_info()
    fit_q.obs_fisher()
    fit_q.log_covar()

    fit_m = Ncm.Fit.factory(Ncm.FitType.NLOPT, "ln-neldermead", lh_m, mset_m, Ncm.FitGradType.NUMDIFF_FORWARD)
    fit_m.run(Ncm.FitRunMsgs.FULL)
    fit_m.log_info()
    fit_m.obs_fisher()
    fit_m.log_covar()

    alphas_q.append(slm_q.props.alpha)
    alphas_err_q.append(fit_q.covar_fparam_sd(0))
    a_s_q.append(slm_q.props.a)
    a_s_err_q.append(fit_q.covar_fparam_sd(1))

    alphas_m.append(slm_m.props.alpha)
    alphas_err_m.append(fit_m.covar_fparam_sd(0))
    a_s_m.append(slm_m.props.a)
    a_s_err_m.append(fit_m.covar_fparam_sd(1))

    mean_time_halo_q.append(mean_time_halo_q_temp)
    std_time_halo_q.append(std_time_halo_q_temp)


    mean_time_halo_m.append(mean_time_halo_m_temp)
    std_time_halo_m.append(std_time_halo_m_temp)

    mean_time_halo.append(mean_time_halo_temp)
    std_time_halo.append(std_time_halo_temp)
    
    # --- Plotting inside the K-loop ---
    size_halo_scaled = np.array(sizes_halo) / 1e6
    x_v = np.linspace(0, 1, 100)
    # Plot Query Phase (_q)
    line_q, = ax_q.plot(x_v, 100 * (slm_q.props.alpha * x_v + slm_q.props.a),linestyle="--")
    ax_q.errorbar(size_halo_scaled, mean_time_halo_q_temp, yerr=2*std_time_halo_q_temp, markersize=4,
                  fmt='o', label=f"k={k}", color=line_q.get_color(), markeredgecolor="black",ecolor="black",capsize=3)

    # Plot Match Phase (_m)
    line_m, = ax_m.plot(x_v, 100 * (slm_m.props.alpha * x_v + slm_m.props.a),linestyle="--")
    ax_m.errorbar(size_halo_scaled, mean_time_halo_m_temp, yerr=2*std_time_halo_m_temp, markersize=4, 
                  fmt='o', label=f"k={k}", color=line_m.get_color(), markeredgecolor="black",ecolor="black",capsize=3)

# --- Finalize Plots AFTER the loop ---
ax_q.set_ylabel("Time (s)",fontsize=14)
ax_q.set_xlabel(r"N query objects ($10^6$)",fontsize=14)
ax_q.tick_params(axis='both', labelsize=12)
ax_q.legend(fontsize=10)
ax_q.grid(True)
ax_q.locator_params(axis='y', nbins=5)
fig_q.tight_layout()
fig_q.savefig("plot_time_x_nobjects_query_prox.pdf")

ax_m.set_ylabel("Time (s)",fontsize=14)
ax_m.set_xlabel(r"N match objects ($10^6$)",fontsize=14)
ax_m.tick_params(axis='both', labelsize=12)
ax_m.legend(fontsize=8)
ax_m.grid(True)
ax_m.locator_params(axis='y', nbins=5)
fig_m.tight_layout()
fig_m.savefig("plot_time_x_nobjects_match_prox.pdf")

plt.show()

knearests = np.array(knearests)
alphas_q = np.array(alphas_q)
alphas_err_q = np.array(alphas_err_q)
a_s_q = np.array(a_s_q)
a_s_err_q = np.array(a_s_err_q)

alphas_m = np.array(alphas_m)
alphas_err_m = np.array(alphas_err_m)
a_s_m = np.array(a_s_m)
a_s_err_m = np.array(a_s_err_m)

In [ ]:
alphas_b = []
alphas_err_b = []
a_s_b = []
a_s_err_b = []


halo_coordinates = {"RA":"RA" , "DEC":"DEC" , "z":"z"}
halo_properties  = {"Mass":"halo_mass","halo_id":"halo_id", "R200":"R200"}
detections_coordinates =  {"RA":"RA" , "DEC":"DEC" , "z":"z"}
cluster_properties  = {"Mass":"cluster_mass","cluster_id":"cluster_id", "R200":"R200"}

fig_b, ax_b = plt.subplots(figsize=(5, 3))

mean_time_halo_b = []
std_time_halo_b  = []

sizes_halo = np.array([35000, 50000, 75000, 100000, 250000, 350000, 500000, 750000, 1000000])
knearests  = [10,20,40,60,80,100]

#sizes_halo = np.array([35000, 50000, 75000])
#knearests  = [10,20]

pbar_k = tqdm(knearests, desc="Processing k Nearest neighboors", unit="k")

for k in knearests:
    pbar_k.set_postfix({"Current k": f"{k}"})
    mean_time_halo_b_temp = []
    std_time_halo_b_temp = []
    
    for size_halo in sizes_halo:
        reduced_halos, clusters = generate_halo_cluster(cosmo,dist,int(2/3*size_halo),int(1/3*size_halo))
        print("Mock data generated")
        # Update the progress bar suffix with the current mass value
    
        halos = sky_match.SkyMatch(
            query_data=reduced_halos, 
            query_coordinates=halo_coordinates,
            match_data=clusters,
            match_coordinates=detections_coordinates
        )
    
        detections = halos.invert_query_match()
        time_match_b = []
        for i in range(5):
            # Match phase
            start_b = time.perf_counter()
            halos_matched = halos.match_2d(cosmo, k, distance_method=sky_match.DistanceMethod.QUERY_RADIUS)
            
            mask_halos = (halos_matched.filter_mask_by_redshift_proximity(sigma_z=0.01) & 
                          halos_matched.filter_mask_by_distance(1))
            unique_halos = halos_matched.select_best(mask=mask_halos, selection_criteria=sky_match.SelectionCriteria.MORE_MASSIVE, more_massive_column='Mass')
            end_b = time.perf_counter()
            time_match_b.append(end_b - start_b)
    
        time_match_b = np.array(time_match_b)
        
        mean_time_halo_b_temp.append(np.mean(time_match_b))
        std_time_halo_b_temp.append(np.std(time_match_b))
    
    mean_time_halo_b_temp = np.array(mean_time_halo_b_temp)
    std_time_halo_b_temp = np.array(std_time_halo_b_temp)
    
    slm_b = PySLineModel()
    slm_b.props.alpha = 0.9
    slm_b.props.a = 0.2
    
    
    mset_b = Ncm.MSet.empty_new()
    mset_b.set(slm_b)
    mset_b.param_set_all_ftype(Ncm.ParamType.FREE)
    mset_b.prepare_fparam_map()

    
    cov_b = Ncm.Matrix.new(len(mean_time_halo_b_temp), len(mean_time_halo_b_temp))
    cov_b.set_zero()
    cov_b.set_diag(Ncm.Vector.new_array(npa_to_seq(np.array(std_time_halo_b_temp)/100)))

    sld_b = PySLineGauss()
    sld_b.set_size(len(mean_time_halo_b_temp))
    sld_b.set_init(True)
    sld_b.use_norma(False)   
    sld_b.xv = Ncm.Vector.new_array(npa_to_seq(sizes_halo/1e6))
    sld_b.set_cov(cov_b)
    sld_b.peek_mean().set_array(npa_to_seq(mean_time_halo_b_temp/100))

    dset_b = Ncm.Dataset.new()
    dset_b.append_data(sld_b)
    lh_b = Ncm.Likelihood.new(dset_b)

    fit_b = Ncm.Fit.factory(Ncm.FitType.NLOPT, "ln-neldermead", lh_b, mset_b, Ncm.FitGradType.NUMDIFF_FORWARD)
    fit_b.run(Ncm.FitRunMsgs.FULL)
    fit_b.log_info()
    fit_b.obs_fisher()
    fit_b.log_covar()

    alphas_b.append(slm_b.props.alpha)
    alphas_err_b.append(fit_b.covar_fparam_sd(0))
    a_s_b.append(slm_b.props.a)
    a_s_err_b.append(fit_b.covar_fparam_sd(1))


    mean_time_halo_b.append(mean_time_halo_b_temp)
    std_time_halo_b.append(std_time_halo_b_temp)

    # --- Plotting inside the K-loop ---
    size_halo_scaled = np.array(sizes_halo) / 1e6
    x_v = np.linspace(0, 1, 100)
    # Plot Query Phase (_q)
    line_b, = ax_b.plot(x_v, 100 * (slm_b.props.alpha * x_v + slm_b.props.a),linestyle="--")
    ax_b.errorbar(size_halo_scaled, mean_time_halo_b_temp, yerr=2*std_time_halo_b_temp, markersize=4,
                  fmt='o', label=f"k={k}", color=line_b.get_color(), markeredgecolor="black",ecolor="black",capsize=3)

# --- Finalize Plots AFTER the loop ---
ax_b.set_ylabel("Time (s)",fontsize=14)
ax_b.set_xlabel(r"N objects ($10^6$)",fontsize=14)
ax_b.tick_params(axis='both', labelsize=12)
ax_b.legend(fontsize=10)
ax_b.grid(True)
ax_b.locator_params(axis='y', nbins=5)
fig_b.tight_layout()
fig_b.savefig("plot_time_x_nobjects_both_prox.pdf")

knearests = np.array(knearests)
alphas_b = np.array(alphas_b)
alphas_err_b = np.array(alphas_err_b)
a_s_b = np.array(a_s_b)
a_s_err_b = np.array(a_s_err_b)

In [ ]:
alphas_q_m_b = [[alphas_q,alphas_err_q,"query"], [alphas_m,alphas_err_m,"match"],[alphas_b,alphas_err_b,"both"]]
for alphas_list in alphas_q_m_b:
    alphas = alphas_list[0]
    alphas_err = alphas_list[1]
    alpha_type = alphas_list[2]
    
    slm_alpha = PySLineModel()
    slm_alpha.props.alpha = 0.9
    slm_alpha.props.a = 0.2
    
    mset = Ncm.MSet.empty_new()
    mset.set(slm_alpha)
    mset.param_set_all_ftype(Ncm.ParamType.FREE)
    mset.prepare_fparam_map()
    
    cov_alpha = Ncm.Matrix.new(len(alphas), len(alphas))
    cov_alpha.set_zero()
    cov_alpha.set_diag(Ncm.Vector.new_array(npa_to_seq(np.array(alphas_err))))
    
    sld_alpha = PySLineGauss()
    sld_alpha.set_size(len(alphas))
    sld_alpha.set_init(True)
    sld_alpha.use_norma(False)
    
    sld_alpha.xv = Ncm.Vector.new_array(npa_to_seq(knearests))
    sld_alpha.set_cov(cov_alpha)
    sld_alpha.peek_mean().set_array(npa_to_seq(alphas))
    
    
    dset = Ncm.Dataset.new()
    dset.append_data(sld_alpha)
    lh = Ncm.Likelihood.new(dset)
    
    fit = Ncm.Fit.factory(Ncm.FitType.NLOPT, "ln-neldermead", lh, mset, Ncm.FitGradType.NUMDIFF_FORWARD)
    fit.run(Ncm.FitRunMsgs.FULL)
    fit.log_info()
    fit.obs_fisher()
    fit.log_covar()

    def bf_alpha_plot(x):
        return 100*(slm_alpha.props.alpha * x + slm_alpha.props.a)
    x_k= np.linspace(10 , 100, 1000)
    # Capture the line object
    plt.figure(figsize=(5,3))
    line, = plt.plot(x_k, bf_alpha_plot(x_k),linestyle="--",color='orange',label="best-fit")
    plt.errorbar(knearests, alphas*100, yerr=2*alphas_err*100, fmt='o', markersize=4, 
              color=line.get_color(),markeredgecolor="black",ecolor="black", capsize=3,label=r"$\alpha$")
    plt.xlabel("k nearest neighboors",fontsize=14)
    plt.ylabel(r"$\alpha_{%s}$" % (alpha_type),fontsize=14)
    plt.grid(True)
    plt.xticks(fontsize=12) 
    plt.yticks(fontsize=12)
    plt.legend(fontsize=12)
    plt.locator_params(axis='y', nbins=5)
    plt.tight_layout()
    plt.savefig(r"alpha_x_k_%s.pdf" %(alpha_type))
    plt.show()

In [ ]:
a_s_q_m_b = [[a_s_q,a_s_err_q,"query"], [a_s_m,a_s_err_m,"match"], [a_s_b,a_s_err_b,"both"]]
for a_s_list in a_s_q_m_b:
    a_s = a_s_list[0]
    a_s_err = a_s_list[1]
    a_s_type = a_s_list[2]
    
    slm_a = PySLineModel()
    slm_a.props.alpha = 0.9
    slm_a.props.a = 0.2
    
    mset = Ncm.MSet.empty_new()
    mset.set(slm_a)
    mset.param_set_all_ftype(Ncm.ParamType.FREE)
    mset.prepare_fparam_map()
    
    cov_a = Ncm.Matrix.new(len(a_s), len(a_s))
    cov_a.set_zero()
    cov_a.set_diag(Ncm.Vector.new_array(npa_to_seq(np.array(a_s_err))))
    
    sld_a = PySLineGauss()
    sld_a.set_size(len(a_s))
    sld_a.set_init(True)
    sld_a.use_norma(False)
    
    sld_a.xv = Ncm.Vector.new_array(npa_to_seq(knearests))
    sld_a.set_cov(cov_a)
    sld_a.peek_mean().set_array(npa_to_seq(a_s))
    
    
    dset = Ncm.Dataset.new()
    dset.append_data(sld_a)
    lh = Ncm.Likelihood.new(dset)
    
    fit = Ncm.Fit.factory(Ncm.FitType.NLOPT, "ln-neldermead", lh, mset, Ncm.FitGradType.NUMDIFF_FORWARD)
    fit.run(Ncm.FitRunMsgs.FULL)
    fit.log_info()
    fit.obs_fisher()
    fit.log_covar()

    def bf_a_plot(x):
        return 100*(slm_a.props.alpha * x + slm_a.props.a)
    x_k= np.linspace(10 , 100, 1000)
    # Capture the line object
    plt.figure(figsize=(5,3))
    line, = plt.plot(x_k, bf_a_plot(x_k),linestyle="--",color='orange',label="best-fit")
    plt.errorbar(knearests, a_s*100, yerr=2*a_s_err*100, fmt='o', markersize=4, 
              color=line.get_color(),markeredgecolor="black",ecolor="black", capsize=3,label=r"$a$")
    plt.xlabel("k nearest neighboors",fontsize=14)
    plt.ylabel(r"$a_{%s}$" % (a_s_type),fontsize=14)
    plt.grid(True)
    plt.xticks(fontsize=12) 
    plt.yticks(fontsize=12)
    plt.legend(fontsize=12)
    plt.locator_params(axis='y', nbins=5)
    plt.tight_layout()
    plt.savefig(r"a_x_k_%s.pdf" %(a_s_type))
    plt.show()

In [ ]:
alphas_q_m_b

[[array([0.52156571, 0.58007887, 0.78354063, 0.83853152, 0.98336291,
         1.11064961]),
  array([0.08351653, 0.08945566, 0.11548516, 0.08560223, 0.07606409,
         0.06774366]),
  'query'],
 [array([0.03735537, 0.03613724, 0.04473911, 0.03394654, 0.04104717,
         0.04222835]),
  array([0.03785674, 0.01883201, 0.04905005, 0.03993682, 0.03414826,
         0.03051171]),
  'match'],
 [array([0.42175777, 0.47608838, 0.58157618, 0.65506464, 0.77652647,
         0.86809839]),
  array([0.03779091, 0.06460318, 0.04783457, 0.06094647, 0.06245051,
         0.0708706 ]),
  'both']]


In [ ]:
a_s_q_m_b

[[array([ 0.00254867,  0.00220498, -0.00509773,  0.01039771,  0.00135335,
          0.00295651]),
  array([0.01921621, 0.0188769 , 0.01704988, 0.03001098, 0.01419254,
         0.01444948]),
  'query'],
 [array([0.01723303, 0.018734  , 0.02146385, 0.03079151, 0.02975358,
         0.03400923]),
  array([0.01228844, 0.00928664, 0.0106465 , 0.02018808, 0.0103584 ,
         0.01043859]),
  'match'],
 [array([-0.00407955, -0.00473982, -0.0060066 , -0.00729866, -0.0090058 ,
         -0.00722654]),
  array([0.00868962, 0.01380092, 0.01298467, 0.00923041, 0.0093753 ,
         0.00960709]),
  'both']]